In [1]:
import pandas as pd
import numpy as np

df = pd.read_parquet('../data/processed/yellow_tripdata_2024-01_clean.parquet')
zones = pd.read_csv('../data/raw/taxi_zone_lookup.csv')

print(f"Loaded: {len(df):,} rows")

Loaded: 2,352,078 rows


In [2]:
# ============================================
# TIME-BASED FEATURES
# ============================================
df['pickup_hour'] = df['tpep_pickup_datetime'].dt.hour
df['pickup_dayofweek'] = df['tpep_pickup_datetime'].dt.dayofweek
df['pickup_day_name'] = df['tpep_pickup_datetime'].dt.day_name()
df['pickup_date'] = df['tpep_pickup_datetime'].dt.date
df['is_weekend'] = df['pickup_dayofweek'].isin([5, 6]).astype(int)
df['day_type'] = np.where(df['is_weekend'] == 1, 'Weekend', 'Weekday')

print("Time features created:")
print(f"  pickup_hour, pickup_dayofweek, pickup_day_name")
print(f"  pickup_date, is_weekend, day_type")

Time features created:
  pickup_hour, pickup_dayofweek, pickup_day_name
  pickup_date, is_weekend, day_type


In [3]:
# ============================================
# PEAK / OFF-PEAK CLASSIFICATION
# ============================================
def classify_peak(hour):
    if 7 <= hour <= 9:
        return 'Morning Peak'
    elif 12 <= hour <= 14:
        return 'Midday'
    elif 17 <= hour <= 19:
        return 'Evening Peak'
    elif 20 <= hour <= 23:
        return 'Night'
    elif 0 <= hour <= 6:
        return 'Early Morning'
    else:
        return 'Off-Peak'

df['peak_period'] = df['pickup_hour'].apply(classify_peak)
df['is_peak'] = df['peak_period'].isin(['Morning Peak', 'Evening Peak']).astype(int)

print("Peak classification:")
print(df['peak_period'].value_counts())

Peak classification:
peak_period
Off-Peak         529559
Evening Peak     493017
Night            454520
Midday           410944
Morning Peak     262368
Early Morning    201670
Name: count, dtype: int64


In [4]:
# ============================================
# SPEED & FARE EFFICIENCY
# ============================================
# Average speed in mph
df['avg_speed_mph'] = df['trip_distance'] / (df['trip_duration_min'] / 60)
df.loc[df['avg_speed_mph'] > 80, 'avg_speed_mph'] = np.nan
df.loc[df['avg_speed_mph'] <= 0, 'avg_speed_mph'] = np.nan

# Fare per mile
df['fare_per_mile'] = df['fare_amount'] / df['trip_distance']
df.loc[df['fare_per_mile'] > 50, 'fare_per_mile'] = np.nan
df.loc[df['fare_per_mile'] <= 0, 'fare_per_mile'] = np.nan

# Fare per minute
df['fare_per_minute'] = df['fare_amount'] / df['trip_duration_min']
df.loc[df['fare_per_minute'] > 10, 'fare_per_minute'] = np.nan
df.loc[df['fare_per_minute'] <= 0, 'fare_per_minute'] = np.nan

print(f"Avg speed:  {df['avg_speed_mph'].mean():.1f} mph")
print(f"Avg $/mile: ${df['fare_per_mile'].mean():.2f}")
print(f"Avg $/min:  ${df['fare_per_minute'].mean():.2f}")

Avg speed:  9.7 mph
Avg $/mile: $8.29
Avg $/min:  $1.22


In [5]:
# ============================================
# ZONE NAME MAPPING
# ============================================
zone_name_map = zones.set_index('LocationID')['Zone'].to_dict()
zone_borough_map = zones.set_index('LocationID')['Borough'].to_dict()

df['PU_zone'] = df['PULocationID'].map(zone_name_map)
df['PU_borough'] = df['PULocationID'].map(zone_borough_map)
df['DO_zone'] = df['DOLocationID'].map(zone_name_map)
df['DO_borough'] = df['DOLocationID'].map(zone_borough_map)

before = len(df)
df = df.dropna(subset=['PU_zone', 'PU_borough', 'DO_zone', 'DO_borough'])
print(f"Zone mapping done. Rows after: {len(df):,} (removed {before - len(df):,})")

Zone mapping done. Rows after: 2,338,354 (removed 13,724)


In [6]:
# ============================================
# CONGESTION FEATURES
# ============================================
# Trip-level congestion flag
df['is_congested_trip'] = (df['avg_speed_mph'] < 10).astype(int)

# Congestion severity (1=free flow to 5=severe)
def congestion_severity(speed):
    if pd.isna(speed):
        return np.nan
    elif speed >= 25:
        return 1
    elif speed >= 18:
        return 2
    elif speed >= 12:
        return 3
    elif speed >= 7:
        return 4
    else:
        return 5

df['congestion_severity'] = df['avg_speed_mph'].apply(congestion_severity)

print("Congestion severity distribution:")
print(df['congestion_severity'].value_counts().sort_index())
print(f"\n% of congested trips (speed < 10 mph): {df['is_congested_trip'].mean()*100:.1f}%")

Congestion severity distribution:
congestion_severity
1.0      13734
2.0      71032
3.0     415257
4.0    1287435
5.0     550598
Name: count, dtype: int64

% of congested trips (speed < 10 mph): 60.7%


In [7]:
# ============================================
# ROUTE IDENTIFIER
# ============================================
df['route'] = df['PU_zone'] + ' -> ' + df['DO_zone']
df['zone_pair'] = df['PULocationID'].astype(str) + '-' + df['DOLocationID'].astype(str)

print(f"Unique routes: {df['route'].nunique():,}")
print(f"Unique zone pairs: {df['zone_pair'].nunique():,}")

Unique routes: 9,183
Unique zone pairs: 9,190


In [8]:
# ============================================
# SAVE ENGINEERED DATA
# ============================================
df.to_parquet('../data/processed/yellow_tripdata_2024-01_engineered.parquet', index=False)

new_cols = [
    'pickup_hour', 'pickup_dayofweek', 'pickup_day_name',
    'pickup_date', 'is_weekend', 'day_type',
    'peak_period', 'is_peak', 'avg_speed_mph',
    'fare_per_mile', 'fare_per_minute',
    'PU_zone', 'PU_borough', 'DO_zone', 'DO_borough',
    'is_congested_trip', 'congestion_severity',
    'route', 'zone_pair'
]

print("=" * 55)
print(f"ENGINEERED DATA SAVED!")
print(f"  Rows: {len(df):,}")
print(f"  Columns: {df.shape[1]}")
print(f"\nNew features added ({len(new_cols)}):")
for col in new_cols:
    print(f"  + {col}")
print("=" * 55)

ENGINEERED DATA SAVED!
  Rows: 2,338,354
  Columns: 39

New features added (19):
  + pickup_hour
  + pickup_dayofweek
  + pickup_day_name
  + pickup_date
  + is_weekend
  + day_type
  + peak_period
  + is_peak
  + avg_speed_mph
  + fare_per_mile
  + fare_per_minute
  + PU_zone
  + PU_borough
  + DO_zone
  + DO_borough
  + is_congested_trip
  + congestion_severity
  + route
  + zone_pair
